In [1]:
%pip install langchain-huggingface langchain-chroma

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#Importing all the required libraries

# Document loading & chunking
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector store
from langchain_chroma import Chroma

# LLM
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
import os

C:\Users\manog\AppData\Local\Temp\ipykernel_43924\2358774711.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
#Loading the book
documents_map = {
    "DIFC_Court_Rules.pdf": "DIFC Court Rules",
    "DFSA_Mkt_Rules_24-25.pdf": "DFSA Markets Rules",
    "DFSA_Gen_module.pdf": "DFSA General Module",
    "DFSA_Gen_Appendix_1.pdf": "DFSA GEN Appendix 1",
    "DFSA_Gen_Appendix_2.pdf": "DFSA GEN Appendix 2",
    "DFSA_Gen_Appendix_3.pdf": "DFSA GEN Appendix 3",
    "DFSA_Gen_Appendix_4.pdf": "DFSA GEN Appendix 4",
    "UAE_Federal_Aml LAW.pdf": "UAE Federal AML Law",
    "DIFC_Data_Protection_Law.pdf": "DIFC Data Protection Law",
}


splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\nArticle", "\nSection", "\n\n", "\n"]
)

all_chunks = []

for filename, doc_name in documents_map.items():
    print(f"Loading {doc_name}...")
    loader = PyPDFLoader(filename)
    pages = loader.load()
    chunks = splitter.split_documents(pages)
    
    # Tag every chunk with which document it came from
    for chunk in chunks:
        chunk.metadata["source_doc"] = doc_name
    
    all_chunks.extend(chunks)
    print(f"  → {len(chunks)} chunks")

print(f"\nTotal chunks across all documents: {len(all_chunks)}")

Loading DIFC Court Rules...
  → 2404 chunks
Loading DFSA Markets Rules...
  → 979 chunks
Loading DFSA General Module...
  → 1145 chunks
Loading DFSA GEN Appendix 1...
  → 73 chunks
Loading DFSA GEN Appendix 2...
  → 39 chunks
Loading DFSA GEN Appendix 3...
  → 231 chunks
Loading DFSA GEN Appendix 4...
  → 67 chunks
Loading UAE Federal AML Law...
  → 112 chunks
Loading DIFC Data Protection Law...
  → 372 chunks

Total chunks across all documents: 5422


In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory="./difc_chroma_db"
)

print("All documents embedded and stored.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

All documents embedded and stored.


In [5]:
# Load existing vectorstore (no re-embedding needed)
vectorstore = Chroma(
    persist_directory="./difc_chroma_db",
    embedding_function=embeddings
)

# Set up Groq
llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model_name="llama-3.1-8b-instant"
)

# Your question
query = "What happens if a defendant fails to respond to a claim?"
# Hybrid retrieval: BM25 keyword search + Chroma semantic search
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

bm25_retriever = BM25Retriever.from_documents(all_chunks)
bm25_retriever.k = 4

semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.5, 0.5]
)

# Retrieve relevant chunks
relevant_chunks = ensemble_retriever.invoke(query)

# Build prompt
context = "\n\n".join([
    f"[{doc.metadata.get('source_doc', 'Unknown Document')} | Page {doc.metadata.get('page', 'N/A')}]: {doc.page_content}"
    for doc in relevant_chunks
])

prompt = f"""You are a compliance assistant for DIFC regulations. 
Answer the question using ONLY the context below.
Cite the document name and page number for every point you make, in the format: Source: Document Name | Page X.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {query}
"""

response = llm.invoke(prompt)
print(response.content)

If a defendant fails to respond to a claim, the claimant may obtain default judgment if Part 13 allows it (Source: DIFC Court Rules | Page 85). 

In the absence of the defendant, the Court may proceed with a trial and strike out the defendant's defence and any counterclaim (Source: DIFC Court Rules | Page 304). The defendant may prove any counterclaim at trial and obtain judgment on his counterclaim and for costs (Source: DIFC Court Rules | Page 304). 

However, if the defendant fails to deal with an allegation in the claim, they shall be taken to admit that allegation (Source: DIFC Court Rules | Page 96).
